In [2]:
import os
import pathlib
from pathlib import Path
import numpy as np
from docx import Document
import requests
import json
from tqdm import tqdm
import pickle
import faiss

In [3]:
def read_docx(filepath: Path) -> str:
    """Чтение текста из .docx файла"""
    doc = Document(filepath)
    text = []
    for para in doc.paragraphs:
        if para.text.strip():
            text.append(para.text)
    return '\n'.join(text)
    

In [4]:
import requests
import numpy as np
import ollama

def get_embedding(model: str, text: str) -> np.ndarray:
    """Получение эмбеддинга через Ollama API"""
    
    response = ollama.embed(model='bge-m3:latest', input=text)
    
    # 2. Проверка на ошибки (404, 500 и т.д.)
    
    return response['embeddings']
        


In [5]:
def process_folder(self, folder_path: str):
    """
    Обработка всех .docx файлов в папке
    
    Args:
        folder_path: путь к папке с .docx файлами
    """
    folder = Path(folder_path)
    
    files = list(folder.glob("*.docx"))
    
    for filepath in tqdm(files, desc="Обработка"):
        text = self.read_docx(filepath)
        
        embedding = self.get_embedding(text)
        if embedding is not None:
            self.embeddings.append(embedding)
            self.texts.append(text[:200])  
            self.filenames.append(filepath.name)
    
    print(f"\nСоздано эмбеддингов: {len(self.embeddings)}")

In [6]:
def save(embeddings, output_path: str = "embeddings.pkl"):
    """Сохранение эмбеддингов в файл"""
    
    with open(output_path, 'wb') as f:
        pickle.dump(embeddings, f)
    
    print(f"Эмбеддинги сохранены в {output_path}")

In [7]:
def create_chunks(text, chunk_size=500, overlap=50):
    """Создает перекрывающиеся чанки из текста"""
    
    if len(text) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        if end < len(text):

            last_period = text.rfind('.', start + chunk_size // 2, end)
            if last_period != -1:
                end = last_period + 1
            else:
                last_space = text.rfind(' ', start + chunk_size // 2, end)
                if last_space != -1:
                    end = last_space + 1
        
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        
        start = end - overlap
        if start >= end:
            start = end
    
    return chunks

In [8]:
FOLDER = "server"
CHUNK_SIZE = 500
OVERLAP = 50

# Собираем все файлы
files = list(Path(FOLDER).glob("*.docx"))
print(f"Найдено файлов: {len(files)}")

Найдено файлов: 9


In [9]:
all_embeddings = []
all_chunks = []
all_info = []

for filepath in files:
    print(f"\nОбработка: {filepath.name}")

    text = read_docx(filepath)
    
    chunks = create_chunks(text, CHUNK_SIZE, OVERLAP)
    
    # Получаем эмбеддинги для каждого чанка
    for i, chunk in enumerate(chunks):
        emb = get_embedding("bge-m3:latest", chunk)
        if emb is not None:
            all_embeddings.append(emb)
            all_chunks.append(chunk)
            all_info.append({
                'file': filepath.name,
                'chunk': i,
                'total_chunks': len(chunks),
                'text_preview': chunk[:100]
            })
    
    print(f"  Добавлено эмбеддингов: {len(all_embeddings)}")



Обработка: По восстановлению срока_не взыскивают.docx
  Добавлено эмбеддингов: 13

Обработка: Возражения_.docx
  Добавлено эмбеддингов: 78

Обработка: Дополнительные вопросы.docx
  Добавлено эмбеддингов: 97

Обработка: ответ на суд запрос_обезличка.docx
  Добавлено эмбеддингов: 103

Обработка: Возражения_обезличка(редактированный).docx
  Добавлено эмбеддингов: 115

Обработка: Возражения суд расходы.docx
  Добавлено эмбеддингов: 136

Обработка: Возражения на заявление о вызскании судебных расходов.docx
  Добавлено эмбеддингов: 165

Обработка: Решение.docx
  Добавлено эмбеддингов: 273

Обработка: Возражения.docx
  Добавлено эмбеддингов: 358


In [10]:
data = {}

for vector_id, text in enumerate(chunks):
    data[vector_id] = {
        "text": text,
        "metadata": {
            "file_id": None,        # если у тебя нет ID документов
            "chunk_index": vector_id
        }
    }

In [11]:
import sqlite3
import pickle

conn = sqlite3.connect("server/user_data.db")
cur = conn.cursor()

with open("chunks.pkl", "rb") as f:
    data = pickle.load(f)

for vector_id, item in data.items():
    # Пока file_id нет, можно ставить 0 или NULL
    cur.execute("""
        INSERT INTO embeddings (file_id, vector_id, available)
        VALUES (?, ?, 1)
    """, (0, vector_id))

conn.commit()
conn.close()


In [12]:
all_embeddings = np.array(all_embeddings).squeeze(1)
all_embeddings.shape

(358, 1024)

In [13]:
all_embeddings[0]

array([-0.00101197,  0.02199645,  0.01073457, ..., -0.00838915,
       -0.04179966,  0.00075291], shape=(1024,))

In [14]:
with open("chunks.pkl", "wb") as f:
    pickle.dump(data, f)

In [21]:
import faiss
def create_index_from_embeddings(embeddings, chunks=None, metadata=None):
    """
    Создание FAISS индекса из готовых эмбеддингов
    
    Args:
        embeddings: список/массив эмбеддингов (numpy array)
        chunks: список текстов чанков (опционально)
        metadata: список метаданных (опционально)
    """
    embeddings = np.array(embeddings, dtype=np.float32)
    
    embedding_dim = embeddings.shape[1]
    
    
    # Создаем индекс (используем IndexFlatL2 для точного поиска)
    index = faiss.IndexFlatL2(embedding_dim)

    print(index)
    
    # Добавляем векторы в индекс
    index.add(embeddings)
    
    # Сохраняем чанки и метаданные
    if chunks is not None:
        chunks = chunks
    if metadata is not None:
        metadata = metadata
    
    print(f"✓ Индекс создан успешно")
    return index

In [22]:
def create_index_from_files(self, embeddings_file, chunks_file=None):
    """
    Создание индекса из файлов с эмбеддингами и чанками
    
    Args:
        embeddings_file: файл с эмбеддингами (.npy или .pkl)
        chunks_file: файл с чанками и метаданными (.pkl)
    """
    
    embeddings = np.load(embeddings_file)
    
    
    # Создаем индекс
    return create_index_from_embeddings(embeddings)

In [2]:
from langchain_community.vectorstores import FAISS
text_embeddings_pairs = list(zip(chunks, all_embeddings))
from langchain_community.embeddings import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model="bge-m3:latest")
vector_db = FAISS.from_embeddings(
    text_embeddings_pairs, 
    embeddings_model# здесь должен быть объект с методом .embed_documents
)

NameError: name 'chunks' is not defined

In [24]:
folder_name = "server/user_faiss"
vector_db.save_local(folder_name)

print(f"✓ Файлы index.faiss и index.pkl успешно созданы в папке {folder_name}")

✓ Файлы index.faiss и index.pkl успешно созданы в папке server/user_faiss


In [25]:
import faiss
import pickle
import os

def save_index(index, index_name, chunks):
    """
    Сохранение FAISS индекса и связанных данных
    
    Args:
        index_path: путь для сохранения индекса (без расширения)
        chunks_path: путь для сохранения чанков (если None - сохранит рядом с индексом)
    """
    
    faiss.write_index(index, "index.faiss")
    print(f"✓ FAISS индекс сохранен в {index}.faiss")
    
    
    chunks_file = f"index.pkl"
    with open(chunks_file, 'wb') as f:
        # ВАЖНО: сохраняем переменную chunks, которую передали в функцию
        pickle.dump(chunks, f)
    
    print(f"✓ Чанки и метаданные сохранены")
    

In [26]:
print(all_chunks[0])

о возмещении судебных расходов,
УСТАНОВИЛ:
истец обратилась в суд с заявлением о возмещении судебных расходов, указывая, что 04.02.2022 Лефортовским районным судом г. Москвы было принято решение, на основании которого исковые требования ХХХХХХвой к Акционерному обществу «Негосударственный пенсионный фонд «ХХХХХХ» (АО «НПФ «ХХХХХХ») о восстановлении срока удовлетворены истцу был восстановлен срок для обращения с заявлением о выплате средств пенсионных накоплений, учтенных в специальной части


In [31]:
from langchain_community.vectorstores import FAISS
text_embeddings_pairs = list(zip(all_chunks, all_embeddings))
faiss_index = FAISS.from_embeddings(text_embeddings_pairs, all_embeddings)
faiss_index.save_local("server/user_faiss")

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [32]:
print(all_embeddings.shape)

(358, 1024)


In [ ]:
import faiss

# Загружаем индеSQLite Viewerкс
index = faiss.read_index("user_faiss/index.faiss")

# Способ 1: Используем метод reconstruct для одного вектора
try:
    # Получаем первый вектор и проверяем его длину
    first_vector = index.reconstruct(0)
    print(f"Размер эмбеддинга: {len(first_vector)}")
    print(f"Тип данных: {first_vector.dtype}")
except:
    pass

# Способ 2: Проверяем атрибуты индекса
print(f"Размерность (d): {index.d}")  # Основной способ!
print(f"Количество векторов: {index.ntotal}")
print(f"Метрика: {'L2' if index.metric_type == faiss.METRIC_L2 else 'Inner Product'}")

# Для IVF индексов
if hasattr(index, 'nlist'):
    print(f"Количество кластеров (nlist): {index.nlist}")

# Для Flat индексов
if isinstance(index, faiss.IndexFlat):
    print("Тип индекса: Flat")

Размер эмбеддинга: 1024
Тип данных: float32
Размерность (d): 1024
Количество векторов: 358
Метрика: L2
Тип индекса: Flat


In [1]:
vector_db.similarity_search(
    query="Что такое обезличка?",
    # kwargs={"k": 5, "score_threshold": 0}
)

NameError: name 'vector_db' is not defined

In [15]:
DB_PATH = "server/user_data.db"
def init_db(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # таблица uploads (виртуальные документы)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS uploads (
            file_id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id TEXT,
            filename TEXT,
            created_at TEXT,
            available INTEGER DEFAULT 1
        )
    """)

    # таблица embeddings (чанки)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS embeddings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_id INTEGER,
            vector_id INTEGER UNIQUE,
            available INTEGER DEFAULT 1
        )
    """)

    conn.commit()
    return conn, cur


In [16]:
def embed_texts(texts):
    # замените на реальную модель embedding
    return np.random.rand(len(texts), 1024).astype("float32")

In [17]:
from datetime import datetime
def index_texts(texts, user_id="user_default"):
    """
    texts: список строк
    user_id: ID пользователя
    """
    conn, cur = init_db()
    pickle_data = {}
    index = faiss.IndexIDMap(faiss.IndexFlatIP(1024))
    vector_id_counter = 0

    for i, text in enumerate(texts):
        filename = f"text_{i+1}"

        # 1️⃣ Добавляем запись в uploads
        cur.execute("""
            INSERT INTO uploads (user_id, filename, created_at, available)
            VALUES (?, ?, ?, 1)
        """, (user_id, filename, datetime.utcnow().isoformat()))
        file_id = cur.lastrowid

        # 2️⃣ Генерация embedding
        vector = embed_texts([text])[0:1]  # shape (1, EMBED_DIM)

        # 3️⃣ vector_id = уникальный ключ
        vector_id = vector_id_counter
        vector_id_counter += 1

        # 4️⃣ Добавляем в embeddings
        cur.execute("""
            INSERT INTO embeddings (file_id, vector_id, available)
            VALUES (?, ?, 1)
        """, (file_id, vector_id))

        # 5️⃣ Добавляем в FAISS
        index.add_with_ids(vector, np.array([vector_id]))

        # 6️⃣ Добавляем в pickle
        pickle_data[vector_id] = {
            "text": text,
            "metadata": {
                "file_id": file_id,
                "chunk_index": 0,
                "filename": filename,
                "user_id": user_id
            }
        }

    # Сохраняем pickle
    pickle_file = "chunks.pkl"
    with open(pickle_file, "wb") as f:
        pickle.dump(pickle_data, f)

    conn.commit()
    conn.close()

    return index, pickle_data


In [18]:
index, pickle_data = index_texts(chunks, user_id="user123")

/tmp/ipykernel_25546/64372739.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  """, (user_id, filename, datetime.utcnow().isoformat()))
